# Convert audit_v11 Timestamped Actions to Simple Left/Right/Camera/Other GT

This notebook programmatically converts the timestamped action annotations produced by `audit_timestamped_tool_interface.ipynb` into a compact per-clip format. It does not manually encode clip annotations; it reads `audit_v11` and the same interface taxonomy/constants used by the audit UI.

In [1]:

import csv
import json
import os
import sys
from pathlib import Path

ROOT_DIR = Path('/shared_data0/weiqiuy/surgent')
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cvs_act.audit_timestamped_tool_interface import (
    CAMERA_ACTION_CODES,
    EXTRA_ACTION_CODES,
    EXTRA_TOOL_TYPES,
    RETRACTION_DIRECTION_OPTIONS,
)

ANNOTATION_ROOT = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1'
AUDIT_V11_DIR = ANNOTATION_ROOT / 'audit_v11'
TAXONOMY_PATH = ANNOTATION_ROOT / 'taxonomy_v10.json'
OUT_DIR = ROOT_DIR / 'notebooks/artifacts/audit_v11_simple_action_gt'
SIMPLE_GT_PATH = OUT_DIR / 'audit_v11_simple_actions.json'
OPTIONS_PATH = OUT_DIR / 'audit_v11_simple_action_options.json'
CSV_PATH = OUT_DIR / 'audit_v11_simple_actions_flat.csv'

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('audit_v11:', AUDIT_V11_DIR)
print('taxonomy:', TAXONOMY_PATH)
print('out:', OUT_DIR)


audit_v11: /shared_data0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1/audit_v11
taxonomy: /shared_data0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1/taxonomy_v10.json
out: /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt


In [2]:

def read_json(path):
    return json.loads(Path(path).read_text())


def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2) + '\n')


def taxonomy_options():
    taxonomy = read_json(TAXONOMY_PATH)
    options = {field: [item['value'] for item in items] for field, items in taxonomy['fields'].items()}
    for code in EXTRA_ACTION_CODES:
        if code not in options['action_code']:
            options['action_code'].append(code)
    for tool in EXTRA_TOOL_TYPES:
        if tool not in options['tool_type']:
            options['tool_type'].append(tool)
    return options


def is_set(value):
    return value not in (None, '', '(not set)', 'Null', 'null')


def frame_or_none(value):
    if value is None:
        return None
    try:
        return int(value)
    except Exception:
        return None


def sorted_segments(rows):
    return sorted(rows, key=lambda row: (row.get('start_frame') is None, row.get('start_frame') or -1, row.get('end_frame') or -1, json.dumps(row, sort_keys=True)))


def load_audit_v11_records():
    records = []
    for path in sorted(AUDIT_V11_DIR.glob('*.json')):
        for record in read_json(path):
            record = dict(record)
            record['_source_path'] = str(path)
            records.append(record)
    return records

records = load_audit_v11_records()
options = taxonomy_options()
print('records:', len(records))
print('taxonomy action codes:', len(options['action_code']))


records: 90
taxonomy action codes: 25


In [3]:

def generic_other_segment(action):
    start = action.get('action_start_frame', action.get('start_frame'))
    end = action.get('action_end_frame', action.get('end_frame'))
    return {
        'start_frame': frame_or_none(start),
        'end_frame': frame_or_none(end),
        'actor_role': action.get('actor_role', '(not set)'),
        'tool_type': action.get('tool_type', '(not set)'),
        'action_code': action.get('action_code', '(not set)'),
        'target_structure': action.get('target_structure', '(not set)'),
        'target_context_1': action.get('target_context_1', '(not set)'),
        'target_context_2': action.get('target_context_2', '(not set)'),
        'description': action.get('one_sentence') or action.get('generated_sentence') or action.get('original_sentence') or '',
        'rank': action.get('rank'),
    }


def convert_record(record):
    coarse = record.get('coarse', {})
    out = {
        'example_id': record.get('example_id'),
        'video_id': record.get('video_id'),
        'criterion': record.get('criterion'),
        'mind_change': record.get('mind_change'),
        'frame_range': [frame_or_none(coarse.get('start_frame')), frame_or_none(coarse.get('end_frame'))],
        'source_path': record.get('_source_path'),
        'left': [],
        'right': [],
        'camera': [],
        'other': [],
    }
    for action in coarse.get('annotation', {}).get('actions_ranked', []):
        rank = action.get('rank')
        actor_role = action.get('actor_role', '(not set)')
        handled = False

        for seg in action.get('left_action_segments') or []:
            out['left'].append({
                'start_frame': frame_or_none(seg.get('start_frame')),
                'end_frame': frame_or_none(seg.get('end_frame')),
                'retraction_direction_code': seg.get('action_code', '(not set)'),
                'changed': seg.get('changed', 'unsure'),
                'start_direction': seg.get('start_direction', 'unsure'),
                'end_direction': seg.get('end_direction', 'unsure'),
                'description': seg.get('description', ''),
                'rank': rank,
            })
            handled = True

        for seg in action.get('right_action_segments') or []:
            triplet = [seg.get('tool_type', '(not set)'), seg.get('action_code', '(not set)'), seg.get('target_structure', '(not set)')]
            out['right'].append({
                'start_frame': frame_or_none(seg.get('start_frame')),
                'end_frame': frame_or_none(seg.get('end_frame')),
                'tool_type': triplet[0],
                'action_code': triplet[1],
                'target_structure': triplet[2],
                'triplet': triplet,
                'target_context_1': seg.get('target_context_1', '(not set)'),
                'target_context_2': seg.get('target_context_2', '(not set)'),
                'description': seg.get('description', ''),
                'rank': rank,
            })
            handled = True

        for seg in action.get('camera_action_segments') or []:
            out['camera'].append({
                'start_frame': frame_or_none(seg.get('start_frame')),
                'end_frame': frame_or_none(seg.get('end_frame')),
                'action_code': seg.get('action_code', seg.get('camera_movement', '(not set)')),
                'camera_movement': seg.get('camera_movement', seg.get('action_code', '(not set)')),
                'description': seg.get('description', ''),
                'rank': rank,
            })
            handled = True

        action_code = action.get('action_code')
        if (not handled and actor_role not in {'left_instrument', 'right_instrument', 'camera'}) or action_code == 'ICG_SWITCH':
            out['other'].append(generic_other_segment(action))

    for key in ['left', 'right', 'camera', 'other']:
        out[key] = sorted_segments(out[key])
    return out

simple_records = [convert_record(record) for record in records]
print('converted:', len(simple_records))
print(json.dumps(simple_records[0], indent=2)[:2000])


converted: 90
{
  "example_id": "00467596-8200-449c-8528-d4816ec2f6a2__C1__avg__c_001950_002400",
  "video_id": "00467596-8200-449c-8528-d4816ec2f6a2",
  "criterion": "C1",
  "mind_change": "unsatisfied->satisfied",
  "frame_range": [
    1950,
    2400
  ],
  "source_path": "/shared_data0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1/audit_v11/00467596-8200-449c-8528-d4816ec2f6a2.json",
  "left": [
    {
      "start_frame": 2040,
      "end_frame": 2160,
      "retraction_direction_code": "RETRACT_MEDIAL_TO_LATERAL",
      "changed": "unsure",
      "start_direction": "unsure",
      "end_direction": "unsure",
      "description": "",
      "rank": 2
    }
  ],
  "right": [
    {
      "start_frame": 1950,
      "end_frame": 2040,
      "tool_type": "Hook",
      "action_code": "DISSECT",
      "target_structure": "CysticArtery",
      "triplet": [
        "Hook",
        "DISSECT",
        "CysticArtery"
      ],
      "target_context_1": "between cyst

In [4]:

def unique_sorted(values):
    return sorted({value for value in values if is_set(value)})

observed_left_retraction_direction_codes = unique_sorted(
    seg['retraction_direction_code']
    for record in simple_records
    for seg in record['left']
)
observed_right_triplets = sorted({
    tuple(seg['triplet'])
    for record in simple_records
    for seg in record['right']
    if all(is_set(value) for value in seg['triplet'])
})
observed_camera_action_codes = unique_sorted(
    seg['action_code']
    for record in simple_records
    for seg in record['camera']
)
observed_other_action_codes = unique_sorted(
    seg['action_code']
    for record in simple_records
    for seg in record['other']
)

simple_options = {
    'source': 'audit_timestamped_tool_interface.py + taxonomy_v10.json + observed audit_v11 GT',
    'interface_options': {
        'retraction_direction_options': RETRACTION_DIRECTION_OPTIONS,
        'camera_action_codes': CAMERA_ACTION_CODES,
        'tool_type': options['tool_type'],
        'action_code': options['action_code'],
        'target_structure': options['target_structure'],
        'target_context': options['target_context'],
    },
    'left_retraction_direction_code_options': observed_left_retraction_direction_codes,
    'right_triplet_options': [list(triplet) for triplet in observed_right_triplets],
    'camera_action_code_options': [code for code in CAMERA_ACTION_CODES if code in set(observed_camera_action_codes)] + [code for code in observed_camera_action_codes if code not in CAMERA_ACTION_CODES],
    'other_action_code_options': observed_other_action_codes,
}

write_json(SIMPLE_GT_PATH, simple_records)
write_json(OPTIONS_PATH, simple_options)
print('left options:', simple_options['left_retraction_direction_code_options'])
print('right triplets:', len(simple_options['right_triplet_options']))
print('camera options:', simple_options['camera_action_code_options'])
print('other options:', simple_options['other_action_code_options'])
print('wrote', SIMPLE_GT_PATH)
print('wrote', OPTIONS_PATH)


left options: ['KEEP_RETRACT_LATERAL', 'KEEP_RETRACT_MEDIAL', 'KEEP_RETRACT_UPWARD', 'RETRACT_LATERAL', 'RETRACT_LATERAL_TO_MEDIAL', 'RETRACT_LATERAL_TO_UPWARD', 'RETRACT_MEDIAL', 'RETRACT_MEDIAL_TO_LATERAL', 'RETRACT_UPWARD_TO_LATERAL']
right triplets: 21
camera options: ['CAMERA_ZOOM_IN', 'CAMERA_ZOOM_OUT', 'CAMERA_REPOSITION', 'CAMERA_UNCERTAIN', 'CAMERA_NO_CHANGE']
other options: ['ICG_SWITCH']
wrote /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt/audit_v11_simple_actions.json
wrote /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt/audit_v11_simple_action_options.json


In [5]:

flat_rows = []
for record in simple_records:
    base = {k: record[k] for k in ['example_id', 'video_id', 'criterion', 'mind_change']}
    base['clip_start_frame'], base['clip_end_frame'] = record['frame_range']
    for actor in ['left', 'right', 'camera', 'other']:
        for seg in record[actor]:
            row = dict(base)
            row['actor'] = actor
            row.update({k: json.dumps(v) if isinstance(v, list) else v for k, v in seg.items()})
            flat_rows.append(row)

fieldnames = sorted({key for row in flat_rows for key in row})
with open(CSV_PATH, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(flat_rows)
print('flat rows:', len(flat_rows))
print('wrote', CSV_PATH)


flat rows: 285
wrote /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt/audit_v11_simple_actions_flat.csv
